In [1]:
import json
import random
import re
from collections import Counter

import pandas as pd

pad_export = '../../../data/H8/llm/geluid_annotaties_predictions.json'
pad_map = '../../../data/H8/llm/zeroshottoets/'
pad_tabel = '../../../tab/H8/8.2_zeroshottoets.csv'

condities = ['few_shot', 'zero_shot']

In [2]:
# ---------------------------------------------------------
# 1. Invoer: nachtelijke gebeurtenissen wijk 3
# ---------------------------------------------------------

with open(pad_export, 'r') as f:
    taken = [t['data'] for t in json.load(f)]

def maak_id(meta):
    bron = meta['source_file'].removesuffix('_final.xml')
    return f"{bron}_p{meta['page']}_i{meta['incident']}"

df_taken = pd.DataFrame({
    'id': [maak_id(t['meta']) for t in taken],
    'text': [t['text'] for t in taken]
})

print("Gebeurtenissen in Label Studio:", len(df_taken))
print("Unieke id's:", df_taken['id'].nunique())

Gebeurtenissen in Label Studio: 15683
Unieke id's: 15683


In [3]:
# de 200 gebeurtenissen waarop de prompt ontwikkeld werd (taken 1-200)
with open(pad_map + 'dev_ids.txt', 'r') as f:
    dev_ids = set(f.read().split())

print("Ontwikkelgebeurtenissen:", len(dev_ids))
print("Waarvan in de export:", len(dev_ids & set(df_taken['id'])))

Ontwikkelgebeurtenissen: 200
Waarvan in de export: 200


In [4]:
invoer = df_taken[~df_taken['id'].isin(dev_ids)].reset_index(drop=True)

print("Gebeurtenissen voor de toets:", len(invoer))

# controle: identiek aan het bestand dat naar het model ging
invoer_model = pd.read_csv(pad_map + 'incidents_d3_nocturnal.csv', keep_default_na=False)
print("Identiek aan invoerbestand:", invoer_model.equals(invoer))

Gebeurtenissen voor de toets: 15483
Identiek aan invoerbestand: True


In [5]:
# ---------------------------------------------------------
# 2. Modelruns (Batches API)
# ---------------------------------------------------------
# !Niet opnieuw draaien
# few-shot:  msgbatch_01LoWtTkaizdREyQ9W4jg23h
# zero-shot: msgbatch_01N7dR7uvyku8BgB6XE1xPVa
# API-sleutel enkel via de omgevingsvariabele ANTHROPIC_API_KEY.

#model = 'claude-haiku-4-5-20251001'
#max_tokens = 1000

In [6]:
#import time
#import anthropic
#
#client = anthropic.Anthropic()
#
#for cond in condities:
#    with open(f'prompts/{cond}.txt', 'r') as f:
#        prompt = f.read()
#
#    # custom_id laat geen '#' toe: rij-index, het id wordt nadien teruggezet
#    verzoeken = []
#    for i, tekst in enumerate(invoer['text']):
#        verzoeken.append({
#            'custom_id': f'r{i}',
#            'params': {
#                'model': model,
#                'max_tokens': max_tokens,
#                'system': prompt,
#                'messages': [{'role': 'user', 'content': tekst}]
#            }
#        })
#
#    batch_id = client.messages.batches.create(requests=verzoeken).id
#    print(cond, batch_id)
#
#    while client.messages.batches.retrieve(batch_id).processing_status != 'ended':
#        time.sleep(60)
#
#    with open(pad_map + f'{cond}.jsonl', 'w') as out:
#        for res in client.messages.batches.results(batch_id):
#            rec = {'id': invoer['id'][int(res.custom_id[1:])], 'status': res.result.type}
#            if res.result.type == 'succeeded':
#                rec['raw'] = res.result.message.content[0].text
#            out.write(json.dumps(rec, ensure_ascii=False) + '\n')

In [7]:
# ---------------------------------------------------------
# 3. Modelantwoorden inlezen en parsen
# ---------------------------------------------------------

ruw = {}
for cond in condities:
    with open(pad_map + f'{cond}.jsonl', 'r') as f:
        ruw[cond] = [json.loads(regel) for regel in f]
    print(cond, "antwoorden:", len(ruw[cond]))

few_shot antwoorden: 15483
zero_shot antwoorden: 15483


In [8]:
blok = re.compile(r'```(?:json)?\s*(.*?)```', re.S)
omheining = re.compile(r'^```(?:json)?\s*|\s*```$')

def lees_json(tekst):
    # soms corrigeert het model zichzelf en volgen er meerdere codeblokken: dan telt het laatste
    blokken = blok.findall(tekst)
    if len(blokken) > 1:
        for b in reversed(blokken):
            try:
                return json.loads(b), True
            except json.JSONDecodeError:
                continue

    s = omheining.sub('', tekst.strip()).strip()
    try:
        return json.loads(s), False
    except json.JSONDecodeError:
        pass

    # terugval: van de eerste [ of { tot de laatste ] of }
    starts = [i for i in (s.find('['), s.find('{')) if i != -1]
    if not starts:
        raise ValueError('geen JSON gevonden')
    einde = max(s.rfind(']'), s.rfind('}')) + 1
    return json.loads(s[min(starts):einde]), False

In [9]:
few_shot_labels = [
    'singing',
    'yelling',
    'doorbell_ringing',
    'playing_inn_music',
    'playing_street_music',
    'scolding',
    'disrupting_nightrest',
    'traffic_noise',
    'making_noise',
    'other_sound'
]

# few-shot: lawaaiklacht = making_noise of disrupting_nightrest
overlast_labels = ['making_noise', 'disrupting_nightrest']

def parse_few_shot(obj):
    if not isinstance(obj, list):
        raise ValueError('geen JSON-array')
    spans = []
    for sp in obj:
        label = str(sp.get('label', '')).strip().lower().replace(' ', '_')
        if label not in few_shot_labels:
            raise ValueError(f'onbekend label: {label}')
        spans.append({**sp, 'label': label})
    overlast = any(sp['label'] in overlast_labels for sp in spans)
    return spans, overlast

# zero-shot: lawaaiklacht = het veld noise_complaint
def parse_zero_shot(obj):
    if not isinstance(obj, dict) or not isinstance(obj.get('noise_complaint'), bool):
        raise ValueError('geen object met spans en noise_complaint')
    return list(obj['spans']), obj['noise_complaint']

parsers = {'few_shot': parse_few_shot, 'zero_shot': parse_zero_shot}

In [10]:
# een antwoord dat niet te parsen valt, telt als 'niets gevonden'
teksten = dict(zip(invoer['id'], invoer['text']))
geparst = {}

for cond in condities:
    geparst[cond] = {}
    fouten = []
    for rec in ruw[cond]:
        out = {'parse_ok': False, 'revised': False, 'spans': [], 'noise_complaint': None}
        try:
            obj, revised = lees_json(rec['raw'])
            spans, overlast = parsers[cond](obj)
            out = {'parse_ok': True, 'revised': revised, 'spans': spans, 'noise_complaint': overlast}
        except Exception as e:
            fouten.append((rec['id'], str(e)[:60]))
        geparst[cond][rec['id']] = out

    print(cond)
    print("  geparst:", sum(r['parse_ok'] for r in geparst[cond].values()), "/", len(geparst[cond]))
    print("  fouten:", fouten)

few_shot
  geparst: 15483 / 15483
  fouten: []
zero_shot
  geparst: 15482 / 15483
  fouten: [('MA#17871_p31_i7', "Expecting ',' delimiter: line 5 column 50 (char 92)")]


In [11]:
for cond in condities:
    r = [x for x in geparst[cond].values() if x['parse_ok']]
    niet_verbatim = sum(
        sp.get('text', '') not in teksten[i]
        for i, x in geparst[cond].items() if x['parse_ok']
        for sp in x['spans']
    )
    print(cond)
    print("  zelfcorrecties:", sum(x['revised'] for x in r))
    print("  gebeurtenissen met geluid:", sum(bool(x['spans']) for x in r))
    print("  gebeurtenissen met lawaaiklacht:", sum(x['noise_complaint'] for x in r))
    print("  spans:", sum(len(x['spans']) for x in r))
    print("  niet-verbatim spans:", niet_verbatim)

few_shot
  zelfcorrecties: 13
  gebeurtenissen met geluid: 2093
  gebeurtenissen met lawaaiklacht: 996
  spans: 3133
  niet-verbatim spans: 81
zero_shot
  zelfcorrecties: 0
  gebeurtenissen met geluid: 2775
  gebeurtenissen met lawaaiklacht: 1232
  spans: 3530
  niet-verbatim spans: 53


In [ ]:
# ---------------------------------------------------------
# 4. Gestratificeerde steekproef
# ---------------------------------------------------------
# strata op de unie van beide condities:
# flagged = minstens één conditie vond geluid of lawaaiklacht (parsefout telt mee)
# rest    = geen van beide vond iets

flagged = set()
for cond in condities:
    for i, r in geparst[cond].items():
        if not r['parse_ok'] or r['spans'] or r['noise_complaint']:
            flagged.add(i)

strata = {
    'flagged': sorted(i for i in invoer['id'] if i in Doe flagged),
    'rest': sorted(i for i in invoer['id'] if i not in flagged)
}

print("Flagged:", len(strata['flagged']))
print("Rest:", len(strata['rest']))

Flagged: 3117
Rest: 12366


In [13]:
seed = 20261105
n_stratum = {'flagged': 200, 'rest': 100}

rng = random.Random(seed)
rijen = []
for naam, ids in strata.items():
    gewicht = len(ids) / n_stratum[naam]
    for i in rng.sample(ids, n_stratum[naam]):
        rijen.append({'id': i, 'stratum': naam, 'weight': gewicht, 'text': teksten[i]})
    print(naam, "gewicht:", round(gewicht, 2))

steekproef = pd.DataFrame(rijen)

# importbestand voor Label Studio: enkel tekst en id, in willekeurige volgorde
taken_ls = [{'data': {'text': r['text'], 'id': r['id']}} for r in rijen]
rng.shuffle(taken_ls)

flagged gewicht: 15.59
rest gewicht: 123.66


In [14]:
# controle: identiek aan de geannoteerde steekproef
steekproef_bestand = pd.read_csv(pad_map + 'sample_300.csv', keep_default_na=False)

print("Identiek aan sample_300.csv:", steekproef_bestand['id'].tolist() == steekproef['id'].tolist())
print("Ontwikkelgebeurtenissen in steekproef:", len(set(steekproef['id']) & dev_ids))

Identiek aan sample_300.csv: True
Ontwikkelgebeurtenissen in steekproef: 0


In [15]:
# ---------------------------------------------------------
# 5. Blinde annotatie
# ---------------------------------------------------------
# lawaaiklacht = de agent omschrijft het incident in termen van geluidsoverlast
# ("lawaai maken" en varianten, of "de (nacht)rust storen" en varianten)

with open(pad_map + 'blind300_export.json', 'r') as f:
    export = json.load(f)

blind = {}
for taak in export:
    anns = [a for a in taak['annotations'] if not a.get('was_cancelled')]
    ann = max(anns, key=lambda a: a.get('updated_at') or a.get('created_at') or '')
    for res in ann['result']:
        if res['from_name'] == 'noise_complaint':
            blind[taak['data']['id']] = res['value']['choices'][0] == 'lawaaiklacht'

print("Geannoteerd:", len(blind))
print("Zelfde id's als steekproef:", set(blind) == set(steekproef['id']))
print("Lawaaiklachten (blind):", sum(blind.values()))

Geannoteerd: 300
Zelfde id's als steekproef: True
Lawaaiklachten (blind): 41


In [16]:
# 14 oordelen die de vooraf vastgelegde definitie niet volgden, achteraf hersteld
correcties = pd.read_csv(pad_map + 'noise_corrections.csv')

hersteld = dict(blind)
for _, c in correcties.iterrows():
    hersteld[c['id']] = bool(c['noise_complaint'])

print("Aangepaste oordelen:", len(correcties))
print("Lawaaiklachten (hersteld):", sum(hersteld.values()))
correcties

Aangepaste oordelen: 14
Lawaaiklachten (hersteld): 51


,id,noise_complaint,reden
0,731#1602_p9_i3,True,"""de nachtrust storde der inwoonders"""
1,731#1608_p102_i5,True,"""nachtrust stoorde"""
2,731#1610_p279_i5,True,"""groot lawijd"""
3,731#1611_p245_i4,True,"""de nachtrust stoorde"""
4,731#1612_p556_i6,True,"""lawijt maakte"""
5,731#1613_p109_i3,True,"""de rust stoorde … met lewijt te maken"""
6,731#1594_p218_i2,True,"""nacht ramoer te maken"", ""nachtrumder"""
7,MA#17633_p80_i2,True,"""nachtrumoer maakte, van aard de rust der inwo..."
8,MA#17875_p314_i1,True,"""hevig lowaaid maakten … de nachtrust der inwo..."
9,MA#17881_p129_i2,True,"""nachtlawijd"""


In [17]:
# ---------------------------------------------------------
# 6. Evaluatie lawaaiklacht
# ---------------------------------------------------------
# teruggewogen naar alle gebeurtenissen met de stratumgewichten;
# 95%-interval uit een gestratificeerde bootstrap

gewichten = dict(zip(steekproef['id'], steekproef['weight']))
stratum = dict(zip(steekproef['id'], steekproef['stratum']))

def scores(ids, gold, cond):
    tp = fp = fn = 0.0
    for i in ids:
        w = gewichten[i]
        g = gold[i]
        p = bool(geparst[cond][i]['noise_complaint'])
        tp += w * (p and g)
        fp += w * (p and not g)
        fn += w * (g and not p)
    precisie = tp / (tp + fp) if tp + fp else float('nan')
    recall = tp / (tp + fn) if tp + fn else float('nan')
    f1 = 2 * precisie * recall / (precisie + recall) if precisie + recall else float('nan')
    return {'precisie': precisie, 'recall': recall, 'F1': f1}

def interval(waarden):
    v = sorted(x for x in waarden if x == x)
    return v[int(0.025 * (len(v) - 1))], v[int(0.975 * (len(v) - 1))]

In [18]:
n_boot = 2000

per_stratum = {}
for s in n_stratum:
    per_stratum[s] = [i for i in steekproef['id'] if stratum[i] == s]

rng = random.Random(seed)
boots = []
for _ in range(n_boot):
    boots.append([i for ids in per_stratum.values() for i in rng.choices(ids, k=len(ids))])

for cond in condities:
    fouten = [i for i in steekproef['id'] if not geparst[cond][i]['parse_ok']]
    print(cond, "parsefouten in steekproef (tellen als niets gevonden):", fouten)

few_shot parsefouten in steekproef (tellen als niets gevonden): []
zero_shot parsefouten in steekproef (tellen als niets gevonden): ['MA#17871_p31_i7']


In [19]:
metrieken = ['precisie', 'recall', 'F1']
ids = list(steekproef['id'])
rijen = []

for versie, gold in [('hersteld', hersteld), ('blind', blind)]:
    punt = {c: scores(ids, gold, c) for c in condities}
    bs = {c: [scores(b, gold, c) for b in boots] for c in condities}
    geschat = sum(gewichten[i] for i in ids if gold[i])

    for c in condities:
        n = Counter((bool(geparst[c][i]['noise_complaint']), gold[i]) for i in ids)
        rij = {
            'versie': versie,
            'conditie': c.replace('_', '-'),
            'TP': n[True, True],
            'FP': n[True, False],
            'FN': n[False, True],
            'geschatte_lawaaiklachten': round(geschat)
        }
        for m in metrieken:
            laag, hoog = interval([b[m] for b in bs[c]])
            rij.update({m: punt[c][m], f'{m}_laag': laag, f'{m}_hoog': hoog})
        rijen.append(rij)

    rij = {'versie': versie, 'conditie': 'verschil zero − few'}
    for m in metrieken:
        laag, hoog = interval([z[m] - f[m] for f, z in zip(bs['few_shot'], bs['zero_shot'])])
        rij.update({m: punt['zero_shot'][m] - punt['few_shot'][m], f'{m}_laag': laag, f'{m}_hoog': hoog})
    rijen.append(rij)

In [20]:
tabel = pd.DataFrame(rijen)

for k in ['TP', 'FP', 'FN', 'geschatte_lawaaiklachten']:
    tabel[k] = tabel[k].astype('Int64')

tabel.round(3).to_csv(pad_tabel, index=False, sep=';', decimal=',')
tabel.style.format(precision=3, decimal=',', thousands='.', na_rep='').hide(axis='index')

versie,conditie,TP,FP,FN,geschatte_lawaaiklachten,precisie,precisie_laag,precisie_hoog,recall,recall_laag,recall_hoog,F1,F1_laag,F1_hoog
hersteld,few-shot,49,20,2,795,"0,710","0,600","0,812","0,961","0,902","1,000","0,817","0,734","0,885"
hersteld,zero-shot,47,37,4,795,"0,560","0,453","0,663","0,922","0,843","0,981","0,696","0,602","0,776"
hersteld,verschil zero − few,,,,,"-0,151","-0,251","-0,046","-0,039","-0,118","0,036","-0,120","-0,203","-0,039"
blind,few-shot,37,32,4,639,"0,536","0,412","0,648","0,902","0,800","0,979","0,673","0,558","0,765"
blind,zero-shot,36,48,5,639,"0,429","0,325","0,533","0,878","0,771","0,971","0,576","0,467","0,672"
blind,verschil zero − few,,,,,"-0,108","-0,189","-0,023","-0,024","-0,128","0,082","-0,097","-0,177","-0,016"


In [21]:
# ---------------------------------------------------------
# 7. Foutanalyse (herstelde versie)
# ---------------------------------------------------------

def spans_tekst(cond, i):
    spans = geparst[cond][i]['spans']
    if cond == 'few_shot':
        return '; '.join(f"{s.get('text', '')} [{s['label']}]" for s in spans)
    return '; '.join(str(s.get('text', '')) for s in spans)

for cond in condities:
    vals = [i for i in ids if geparst[cond][i]['noise_complaint'] and not hersteld[i]]
    print(f"\n{cond} – vals positief ({len(vals)})")
    for i in sorted(vals):
        print(f"  {i:<20} {spans_tekst(cond, i)[:110]}")


few_shot – vals positief (20)
  731#1587_p151_i3     wanorders gemaakt [making_noise]
  731#1587_p25_i3      desordre [making_noise]
  731#1587_p44_i4      geroepen [yelling]; twist te maken [making_noise]
  731#1593_p64_i5      wanorders op de Groenplaat maakte [making_noise]
  731#1594_p155_i4     opver maekte [making_noise]
  731#1610_p501_i1     opstand maakte [making_noise]
  731#1614_p405_i5     wanorders [making_noise]
  731#1617_p70_i3      eene groote verzameling van volk teweegbracht [making_noise]
  MA#17608_p136_i4     er geslagen [making_noise]
  MA#17615_p177_i3     nedetuisten [making_noise]
  MA#17615_p44_i1      groot gar op levorde [making_noise]
  MA#17617_p167_i5     luid = ruchtig spreken [making_noise]
  MA#17624_p53_i3      tuidkeels turstte [making_noise]
  MA#17630_p291_i3     nachtgerucht maaktie [making_noise]; aan de bel harer woning te trekken [doorbell_ringing]
  MA#17632_p110_i3     luidruchtig gesprek [making_noise]
  MA#17634_p263_i2     smeldeg te heb

In [22]:
for cond in condities:
    gemist = [i for i in ids if hersteld[i] and not geparst[cond][i]['noise_complaint']]
    print(f"\n{cond} – gemist ({len(gemist)})")
    for i in sorted(gemist):
        print(f"  {i:<20} {spans_tekst(cond, i)[:110]}")


few_shot – gemist (2)
  MA#17624_p69_i3      roepende [yelling]; aan de bel hok [doorbell_ringing]
  MA#17871_p31_i7      

zero_shot – gemist (4)
  731#1613_p280_i2     nachtrust gestoord
  MA#17621_p230_i2     lijwijt maakte
  MA#17644_p68_i4      hobaal de moker
  MA#17871_p31_i7      
